In [ ]:
import weaviate
from weaviate.classes.config import Configure, Property, DataType
from weaviate.classes.query import Filter

In [ ]:
# 1. 准备数据
items = [
    {
        "type": "phone",
        "id": "用户A",
        "content": "用户A的手机号是13800001234"
    },
    {
        "type": "phone",
        "id": "用户B",
        "content": "用户B的手机号是13800005678"
    },
    {
        "type": "order",
        "id": "订单1001",
        "content": "订单1001编号203011010001"
    },
    {
        "type": "order",
        "id": "订单1002",
        "content": "订单1002编号203011010123"
    }
]

collection_name = "NumberItem"

In [ ]:
# 2. 创建 / 获取集合
def get_or_create_collection(client:weaviate.WeaviateClient):

    if client.collections.exists(collection_name):
        return client.collections.get(collection_name)

    return client.collections.create(
        name=collection_name,
        vectorizer_config=Configure.Vectorizer.none(),  # 自己提供向量，不用内置模型
        properties=[
            Property(name="type", data_type=DataType.TEXT),
            Property(name="item_id", data_type=DataType.TEXT),
            Property(name="number", data_type=DataType.NUMBER),
        ]
    )

In [ ]:
# 3. 插入数据
def insert_data(numbers_collection):
# 向 batch 中添加数据。当添加的数据达到 200 条时，底层会自动执行一次提交；当离开 with 块时，即使最后不足 200 条的数据也会被自动提交，并清理资源。
    with numbers_collection.batch.fixed_size(batch_size=200) as batch:

        for item in items:

            batch.add_object(

                vector=[float(item["number"])],
                properties={
                    "type": item["type"],
                    "item_id": item["id"],
                    "number": item["number"],
                },
                
            )

In [ ]:
# 4. 查询：全量向量搜索
def query_near_vector_all(numbers_collection, query_number):
    """纯向量搜索，不过滤"""
    response = numbers_collection.query.near_vector(
        near_vector=[float(query_number)],
        limit=5,
    )

    print(f"查询数字: {query_number}")
    print("Top-5 最相近的记录（全部类型）：")

    for i, item in enumerate(response.objects, start=1):
        props = item.properties
        print(
            f"第{i}条：类型={props['type']}，编号={props['item_id']}，"
            f"数字={props['number']}，距离={item.metadata.distance:.4f}"
        )

In [ ]:
# 5. 查询：向量 + 过滤（混合搜索）
def query_near_vector_filtered(numbers_collection, query_number):

    where_filter = Filter.by_property("type").equal("order")

    response = numbers_collection.query.near_vector(
        near_vector=[float(query_number)],
        limit=5,
        filters=where_filter,
    )

    print(f"查询数字: {query_number}")
    print("Top-5 最相近的记录（仅 order 类型）：")

    for i, item in enumerate(response.objects, start=1):
        props = item.properties
        print(
            f"第{i}条：类型={props['type']}，编号={props['item_id']},"
            f"数字={props['number']}，距离={item.metadata.distance:.4f}"
        )

In [ ]:
# 6. 主流程
def main():
    with weaviate.connect_to_local() as client:
        numbers = get_or_create_collection(client)

        insert_data(numbers)

        query_number = 205001020500

        # 二选一，按需打开
        query_near_vector_all(numbers, query_number)
        # query_near_vector_filtered(numbers, query_number)


main()